### Data Fetching

In [1]:
import psycopg2
import pandas as pd

In [2]:
def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_17940\63475477.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7760 rows from 'extraction'


In [5]:
keywords = ["Radio"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    # (df['status_id'] == 2) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df

,filename,workorder_id,json_data
821,SC_PM_QTR_Radio_NA_29.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1234,SC_PM_QTR_Radio_NA_2.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1244,SC_PM_QTR_Radio_NA_30.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1262,SC_PM_QTR_Radio_NA_31.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1280,SC_PM_QTR_Radio_NA_32.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...
7701,SC_PM_QTR_Radio_NA_24.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7714,SC_PM_QTR_Radio_NA_25.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7741,SC_PM_QTR_Radio_NA_26.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7758,SC_PM_QTR_Radio_NA_27.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."


In [7]:
import pandas as pd

def rename_keys(d):
    if not isinstance(d, dict):
        return d
    
    mapping = {
        'radio_system': 'radio',
    }

    return {mapping.get(k, k): v for k, v in d.items()}

df['json_data'] = df['json_data'].apply(rename_keys)

valid_json = df['json_data'][df['json_data'].apply(lambda x: isinstance(x, dict))]
all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))


['notification', 'radio', 'work_order']


In [8]:
#check the full content of json data for the first few rows
df

,filename,workorder_id,json_data
821,SC_PM_QTR_Radio_NA_29.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1234,SC_PM_QTR_Radio_NA_2.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1244,SC_PM_QTR_Radio_NA_30.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1262,SC_PM_QTR_Radio_NA_31.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
1280,SC_PM_QTR_Radio_NA_32.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...
7701,SC_PM_QTR_Radio_NA_24.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7714,SC_PM_QTR_Radio_NA_25.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7741,SC_PM_QTR_Radio_NA_26.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."
7758,SC_PM_QTR_Radio_NA_27.pdf,NaN,"{'notification': {'notification_no': 'NA', 'no..."


In [9]:
import pandas as pd
import json

df_radio = df.copy(deep=True)

valid_mask = df_radio['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_radio[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename']
    }

    radio_data = row['json_data'].get('radio', {})

    for key, value in radio_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_radio = pd.DataFrame(rows)

df_radio = df_radio.drop(
    columns=["pm_order_no", "reference_document_no", "reference_document"],
    errors="ignore"
)

df_radio.rename(columns={
    'performed_by': 'technician_id',
    'verified_by': 'supervisor_id',
}, inplace=True)

df_radio.head()

,workorder_id,filename,station,date_time,procedures,comment_recommendation,technician_id,supervisor_id
0,NaN,SC_PM_QTR_Radio_NA_29.pdf,MAH,03/10/2022,"{""safety_and_preparation"": {""a"": {""procedure"":...",よい おい、,ROHAIZAN,YUNUS AZHAP
1,NaN,SC_PM_QTR_Radio_NA_2.pdf,RAN,07/04/2022,"{""safety_and_preparation"": {""a"": {""procedure"":...",Coverage Ok.,MOHD BAZIF,AZHAR VUN
2,NaN,SC_PM_QTR_Radio_NA_30.pdf,MKY,09/10/2022,"{""safety_and_preparation"": {""a"": {""procedure"":...",all equstment in good conditiong,LAZIM,AZHAR
3,NaN,SC_PM_QTR_Radio_NA_31.pdf,TSA,11/10/12,"{""safety_and_preparation"": {""a"": {""procedure"":...",All connection and radio test in good condition.,7110,7111
4,NaN,SC_PM_QTR_Radio_NA_32.pdf,11,05/10/2022,"{""safety_and_preparation"": {""a"": {""procedure"":...",NA,Technio,AZHAR


In [10]:
def flatten_procedures(proc_data):
    flat_data = {}
    
    if isinstance(proc_data, str):
        try:
            proc_data = json.loads(proc_data)
        except:
            return {}
            
    if not isinstance(proc_data, dict): 
        return flat_data

    def walk(d, parent_key=''):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            
            if isinstance(v, dict):
                if 'status' in v:
                    flat_data[f"{new_key}.status"] = v.get('status')
                    flat_data[f"{new_key}.remarks"] = v.get('remarks')
                else:
                    walk(v, new_key)
            else:
                flat_data[new_key] = v

    walk(proc_data)
    return flat_data

def parse_json(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return {}
    return x

df_radio["procedures"] = df_radio["procedures"].apply(parse_json)

df_proc_flat = pd.DataFrame(
    df_radio["procedures"].apply(flatten_procedures).tolist(), 
    index=df_radio.index
)

df_final = pd.concat([
    df_radio.drop(columns=['procedures']), 
    df_proc_flat,
], axis=1)

print(df_final.columns)

Index(['workorder_id', 'filename', 'station', 'date_time',
       'comment_recommendation', 'technician_id', 'supervisor_id',
       'safety_and_preparation.a.status', 'safety_and_preparation.a.remarks',
       'safety_and_preparation.b.status', 'safety_and_preparation.b.remarks',
       'remote_site_unit.a.status', 'remote_site_unit.a.remarks',
       'remote_site_unit.b.status', 'remote_site_unit.b.remarks',
       'remote_site_unit.c.status', 'remote_site_unit.c.remarks',
       'remote_site_unit.d.status', 'remote_site_unit.d.remarks',
       'duplexer.a.status', 'duplexer.a.remarks', 'duplexer.b.status',
       'duplexer.b.remarks', 'duplexer.c.status', 'duplexer.c.remarks',
       'gps_frequency_reference.a.status', 'gps_frequency_reference.a.remarks',
       'gps_frequency_reference.b.status', 'gps_frequency_reference.b.remarks',
       'gps_frequency_reference.c.status', 'gps_frequency_reference.c.remarks',
       'front_panel.a.status', 'front_panel.a.remarks', 'front_panel.b.

In [11]:
output_file = f"../../output/snc/radio.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='radio', index=False)

print(f"Saved excel to {output_file}")

Saved excel to ../../output/snc/radio.xlsx
